# Volume 8, Chapter 7: Matrix Lie Groups (Python companion)

The exponential at work: splitting formulas, quaternions versus rotation matrices and Euler angles, gimbal lock, and interpolation of rotations by one-parameter subgroups.

*Before each solution, predict what the output will look like.*

## Exercises

**P7.1 (★) Lie's product formula and the exponential, numerically.** (a) For random $X,Y\in M_3(\R)$, check that $(e^{X/m}e^{Y/m})^m\to e^{X+Y}$ with error of order $\frac1m$, and that the commutator formula of Proposition 7.4 converges to $e^{[X,Y]}$. (b) Using the improved splitting $e^{X/2m}e^{Y/m}e^{X/2m}$ (Strang splitting), observe an error of order $\frac1{m^2}$. Splitting methods of this kind integrate the Schrödinger equation and molecular dynamics: the exact flow of $X+Y$ is unknown, but those of $X$ and $Y$ separately are easy.

**P7.2 (★★) CS application: three ways to store an orientation.** A rigid body in a game or a robot has an orientation in $SO(3)$. (a) Implement rotation matrices, unit quaternions (via the double cover $SU(2)\to SO(3)$ of Theorem 7.8) and Euler angles (yaw, pitch, roll), with conversions between them. (b) Compose a million small random rotations in single precision, storing the result as a matrix or as a quaternion, and measure the drift away from $SO(3)$ (or from the unit sphere), with and without renormalization. (c) *Gimbal lock*: show that the map from Euler angles to $SO(3)$ has a differential of rank $2$ when the pitch is $\pm\frac\pi2$, so that near this attitude some small rotations require huge changes of the angles. Find one.

**P7.3 (★★) CS application: interpolating rotations.** To animate a camera from orientation $R_0$ to $R_1$, one wants a path at constant angular speed. (a) Show that *slerp*, $q(t)=q_0(q_0^{-1}q_1)^t=q_0e^{t\log(q_0^{-1}q_1)}$, is a one-parameter subgroup translated to $q_0$, and that it gives constant angular speed. (b) Compare with linear interpolation of matrices (which leaves $SO(3)$) and of quaternions followed by normalization (``nlerp''), and with linear interpolation of Euler angles. (c) Explain why slerp must first replace $q_1$ by $-q_1$ when $\langle q_0,q_1\rangle<0$, using the double cover.

## Solutions

### P7.1

In [ ]:
import numpy as np
from scipy.linalg import expm

rng = np.random.default_rng(0)
X, Y = rng.standard_normal((3, 3)), rng.standard_normal((3, 3))
E = expm(X + Y); C = expm(X @ Y - Y @ X)
print("   m     Lie product    Strang splitting   commutator formula")
for m in (10, 100, 1000, 10000):
    lie = np.linalg.matrix_power(expm(X / m) @ expm(Y / m), m)
    strang = np.linalg.matrix_power(expm(X / (2 * m)) @ expm(Y / m) @ expm(X / (2 * m)), m)
    comm = np.linalg.matrix_power(expm(X / m) @ expm(Y / m) @ expm(-X / m) @ expm(-Y / m), m * m) if m <= 1000 else None
    e3 = np.linalg.norm(comm - C) if comm is not None else float("nan")
    print(f"{m:6d}   {np.linalg.norm(lie - E):.3e}      {np.linalg.norm(strang - E):.3e}          {e3:.3e}")

Each tenfold increase of $m$ divides the error of Lie's formula by about $10$ (order $\frac1m$, the second-order term $\frac1{2m^2}[X,Y]$ accumulating $m$ times) and that of Strang's symmetric splitting by about $100$ (the symmetric product has no second-order error, by the symmetry $m\to-m$). The commutator formula also converges at rate $\frac1m$. Symmetric splittings are the workhorse of numerical quantum mechanics (split-step Fourier methods), where $X$ is the kinetic part, diagonal in Fourier space, and $Y$ the potential, diagonal in position space; the leapfrog method of the Chapter 2 notebook is the same idea.

### P7.2

In [ ]:
def qmul(p, q):
    a1, b1, c1, d1 = p; a2, b2, c2, d2 = q
    return np.array([a1*a2 - b1*b2 - c1*c2 - d1*d2, a1*b2 + b1*a2 + c1*d2 - d1*c2,
                     a1*c2 - b1*d2 + c1*a2 + d1*b2, a1*d2 + b1*c2 - c1*b2 + d1*a2], dtype=p.dtype)

def q_to_R(q):
    a, b, c, d = q / np.linalg.norm(q)
    return np.array([[a*a + b*b - c*c - d*d, 2*(b*c - a*d), 2*(b*d + a*c)],
                     [2*(b*c + a*d), a*a - b*b + c*c - d*d, 2*(c*d - a*b)],
                     [2*(b*d - a*c), 2*(c*d + a*b), a*a - b*b - c*c + d*d]])

def axis_angle_q(u, th, dtype=np.float64):
    u = np.asarray(u, float) / np.linalg.norm(u)
    return np.r_[np.cos(th / 2), np.sin(th / 2) * u].astype(dtype)

def euler_R(yaw, pitch, roll):
    cz, sz, cy, sy, cx, sx = np.cos(yaw), np.sin(yaw), np.cos(pitch), np.sin(pitch), np.cos(roll), np.sin(roll)
    Rz = np.array([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]])
    Ry = np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]])
    Rx = np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]])
    return Rz @ Ry @ Rx

# check the double cover: q and -q give the same matrix, and q_to_R is a homomorphism
p, q = axis_angle_q([1, 2, 3], 0.7), axis_angle_q([0, 1, -1], 2.1)
print("R(q) = R(-q):", np.allclose(q_to_R(q), q_to_R(-q)), "  R(pq) = R(p)R(q):", np.allclose(q_to_R(qmul(p, q)), q_to_R(p) @ q_to_R(q)))

# (b) drift in single precision
N = 1_000_000
rng = np.random.default_rng(2)
axes = rng.standard_normal((N, 3)).astype(np.float32)
angles = (0.01 * rng.standard_normal(N)).astype(np.float32)
Rm = np.eye(3, dtype=np.float32); qq = np.array([1, 0, 0, 0], dtype=np.float32)
Rn = np.eye(3, dtype=np.float32); qn = qq.copy()
for k in range(N):
    dq = axis_angle_q(axes[k], angles[k], np.float32)
    dR = q_to_R(dq.astype(np.float64)).astype(np.float32)
    Rm = Rm @ dR; qq = qmul(qq, dq)
    Rn = Rn @ dR; qn = qmul(qn, dq)
    if k % 1000 == 999:
        qn /= np.linalg.norm(qn)
        U, _, Vt = np.linalg.svd(Rn.astype(np.float64)); Rn = (U @ Vt).astype(np.float32)
print(f"matrix:     ||R^T R - I|| = {np.linalg.norm(Rm.T.astype(float) @ Rm - np.eye(3)):.2e}   (renormalized every 1000 steps: {np.linalg.norm(Rn.T.astype(float) @ Rn - np.eye(3)):.2e})")
print(f"quaternion: | |q| - 1 |   = {abs(np.linalg.norm(qq.astype(float)) - 1):.2e}   (renormalized: {abs(np.linalg.norm(qn.astype(float)) - 1):.2e})")

In [ ]:
# (c) gimbal lock: rank of the differential of (yaw, pitch, roll) -> SO(3)
def jac(angles, h=1e-6):
    cols = []
    for i in range(3):
        e = np.zeros(3); e[i] = h
        cols.append(((euler_R(*(angles + e)) - euler_R(*(angles - e))) / (2 * h)).ravel())
    return np.array(cols).T

for pitch in (0.0, 1.0, 1.5, np.pi / 2):
    s = np.linalg.svd(jac(np.array([0.3, pitch, -0.4])), compute_uv=False)
    print(f"pitch = {pitch:.4f}: singular values of the differential {np.round(s, 6)}")

# at pitch = pi/2, try to reach a small rotation about the body's... x-axis of the world
target = euler_R(0.3, np.pi / 2 - 1e-3, -0.4) @ np.eye(3)
small = q_to_R(axis_angle_q([1, 0, 0], 0.01))
from scipy.optimize import least_squares
start = np.array([0.3, np.pi / 2 - 1e-3, -0.4])
sol = least_squares(lambda a: (euler_R(*a) - small @ euler_R(*start)).ravel(), start)
print("a rotation by 0.01 rad about the world x-axis changes the Euler angles by", np.round(sol.x - start, 4))

(a)--(b) The check confirms that $q$ and $-q$ give the same rotation and that $q\mapsto R_q$ is a homomorphism: Theorem 7.8 in code. Over a million single-precision compositions the matrix drifts away from $SO(3)$ and the quaternion away from norm $1$; renormalizing a quaternion is one division, while projecting a matrix back onto $SO(3)$ needs an SVD (the polar decomposition, Volume S1). This, with the four numbers instead of nine, is why orientation filters in phones and drones store quaternions.

(c) At pitch $\frac\pi2$ the smallest singular value of the differential is $0$: yaw and roll turn about the same axis, so the three angles only span a $2$-dimensional set of directions. Near this attitude a rotation by $0.01$ radian about the world $x$-axis requires changes of yaw and roll of more than a radian each: the angles spin wildly while the body barely moves. This is gimbal lock. The inertial platforms of the Apollo spacecraft were mounted on three physical gimbals and had to stay away from such attitudes, which is why Michael Collins, according to the Apollo 11 transcript, asked mission control for a fourth gimbal for Christmas; the topological reason is that no map from a $3$-torus of angles can be a local diffeomorphism onto $SO(3)$ everywhere, since such a map would be a covering (it is proper), and coverings of $SO(3)$ have $1$ or $2$ sheets with total spaces $SO(3)$ or $S^3$, not $T^3$.

### P7.3

In [ ]:
def qlog_unit(q):
    a, v = q[0], q[1:]
    nv = np.linalg.norm(v)
    return np.zeros(3) if nv < 1e-15 else np.arctan2(nv, a) * v / nv     # q = exp(theta * u) with theta = half angle

def qexp_pure(w):
    th = np.linalg.norm(w)
    return np.r_[np.cos(th), (np.sin(th) / th) * w if th > 1e-15 else w]

def qinv(q):
    return np.r_[q[0], -q[1:]] / (q @ q)

def slerp(q0, q1, t):
    if q0 @ q1 < 0:
        q1 = -q1
    return qmul(q0, qexp_pure(t * qlog_unit(qmul(qinv(q0), q1))))

def angle_between(R, S):
    return np.arccos(np.clip((np.trace(R.T @ S) - 1) / 2, -1, 1))

q0 = axis_angle_q([0, 0, 1], 0.2); q1 = axis_angle_q([1, 1, 0], 2.8)
R0, R1 = q_to_R(q0), q_to_R(q1)
ts = np.linspace(0, 1, 11)
methods = {
    "slerp": [q_to_R(slerp(q0, q1, t)) for t in ts],
    "nlerp": [q_to_R(((1 - t) * q0 + t * (q1 if q0 @ q1 >= 0 else -q1))) for t in ts],
    "matrix lerp": [(1 - t) * R0 + t * R1 for t in ts],
}
print(f"total angle between R0 and R1: {angle_between(R0, R1):.4f} rad")
for name, Rs in methods.items():
    if name == "matrix lerp":
        dets = [np.linalg.det(R) for R in Rs]
        print(f"{name:12s}: det along the path from {min(dets):.3f} to {max(dets):.3f}  (not rotations)")
        continue
    steps = [angle_between(Rs[i], Rs[i + 1]) for i in range(10)]
    print(f"{name:12s}: angle per step from {min(steps):.4f} to {max(steps):.4f}")
# the double cover: interpolating to -q1 instead of q1 takes the long way round
long = [q_to_R(qmul(q0, qexp_pure(t * qlog_unit(qmul(qinv(q0), -q1 if q0 @ q1 >= 0 else q1))))) for t in ts]
print(f"long way: total angle travelled = {sum(angle_between(long[i], long[i+1]) for i in range(10)):.4f} rad"
      f"  vs short way {sum(angle_between(methods['slerp'][i], methods['slerp'][i+1]) for i in range(10)):.4f} rad")

(a) $q_0^{-1}q_1$ is a unit quaternion $e^{\theta u}$, with $u$ a unit pure quaternion, and $t\mapsto e^{t\theta u}$ is a one-parameter subgroup of $SU(2)$ (Proposition 7.3); its image in $SO(3)$ is the rotation about $u$ by the angle $2\theta t$, which has constant angular speed. Translating by $q_0$ (an isometry of the bi-invariant metric) keeps the speed constant: slerp is a geodesic of $SO(3)$. The output confirms equal angles at every step.

(b) Normalized linear interpolation stays in $SO(3)$ but its speed varies (faster in the middle); matrix interpolation leaves $SO(3)$ altogether, its determinant dropping well below $1$ (it shrinks the object). Interpolating Euler angles, not shown, gives paths that depend on the chosen axes and behave badly near gimbal lock.

(c) $q_1$ and $-q_1$ give the same rotation, but the paths from $q_0$ to them in $S^3$ project to two different paths in $SO(3)$, one turning by $2\theta$ and the other by $2\pi-2\theta$ the other way round; together they form the nontrivial loop of Theorem 7.8. Choosing the sign with $\langle q_0,q_1\rangle\ge0$ selects the short one, as the last line shows.